In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from pathlib import Path

# --- 1) Caminhos
DATA = Path("../data/interim")
DATA_ANALYSIS = Path("../data/analysis")
DATA_FINAL = Path("../data/final")

In [2]:
# --- 2) Leitura e padronização de nomes (evita colisões no merge)
companies = pd.read_parquet(DATA / "ns_resolver/companies.parquet").drop(columns=["longBusinessSummary"])  # 1 linha por empresa (tem 'apex')
display(companies.head(1))
dns_status = (
    pd.read_parquet(DATA / "ns_resolver/dns_status.parquet")
      .rename(columns={"domain": "apex", "status": "dns_status"})
      .drop_duplicates("apex")
)
# DNSs não resolvidos
display(dns_status.head(1))
ns_hosts = (
    pd.read_parquet(DATA / "ns_resolver/ns_hosts.parquet")
      .rename(columns={"domain": "apex"})
      .drop_duplicates(["apex", "ns"])
)
display(ns_hosts.head(1))
ns_ips = (
    pd.read_parquet(DATA / "ns_resolver/ns_ips.parquet")
      .rename(columns={"rtype": "ip_type", "status": "ip_lookup_status"})
)
display(ns_ips.head(1))
ip_asn = (
    pd.read_parquet(DATA / "asn_resolver/ip_asn.parquet")
      .rename(columns={"cc": "prefix_cc", "registry": "prefix_registry", "status": "asn_status"})
      .drop_duplicates("ip")
)
display(ip_asn.head(1))
asn_org = (
    pd.read_parquet(DATA / "asn_resolver" / "asn_org_info.parquet")
      [["asn", "handle", "org", "country_code"]]
      .rename(columns={"handle": "as_handle", "org": "as_org", "country_code": "as_country_code"})
      .drop_duplicates("asn")
)
display(asn_org.head(1))

,rank,company_name,ticker_symbol,revenue_ttm,price_usd,company_country,website,sector,industry,country,city,yf_status,apex
0,1,Amazon,AMZN,742776000000,248.0,United States,https://www.amazon.com,Consumer Cyclical,Internet Retail,United States,Seattle,success,amazon.com


,apex,dns_status
498,chinacoalenergy.com,nxdomain


,apex,ns,ns_apex
0,apple.com,a.ns.apple.com,apple.com


,ns,ip_lookup_status,ip,ip_type
0,a.ns.apple.com,ok,17.253.200.1,A


,ip,asn,asn_all,prefix,prefix_cc,prefix_registry,asn_status
0,2600:9000:5302:fd00::1,16509,16509,2600:9000:5302::/47,US,arin,ok


,asn,as_handle,as_org,as_country_code
0,16509,AMAZON-02,"Amazon.com, Inc.",US


In [3]:
# --- 3) Encadeamento: empresa -> NS -> IP -> ASN -> organização
full = (
    companies
    .merge(dns_status, on="apex", how="left", validate="many_to_one")
    .merge(ns_hosts,   on="apex", how="left")
    .merge(ns_ips,     on="ns",   how="left")
    .merge(ip_asn,     on="ip",   how="left", validate="many_to_one")
    .merge(asn_org,    on="asn",  how="left", validate="many_to_one")
)

In [4]:
# --- 4) Status de DNS consolidado
full.loc[full["apex"].isna(), "dns_status"] = "no_website"
full["dns_status"] = full["dns_status"].fillna(
    full["ns"].notna().map({True: "ok", False: "not_resolved"})
)

In [5]:
# --- 5) Conferências
assert full["rank"].nunique() == companies["rank"].nunique(), "empresas perdidas no merge"
print(f"{len(companies)} empresas -> {len(full)} linhas")
print("\n", full["dns_status"].value_counts())
print("\nlinhas com IP sem ASN:", (full["ip"].notna() & full["asn"].isna()).sum())
print("\n", full.loc[full["asn"].notna(), ["as_handle", "as_org", "as_country_code"]].isna().sum())


10664 empresas -> 83787 linhas

 dns_status
ok            83415
no_website      343
no_ns            18
timeout           9
nxdomain          2
Name: count, dtype: int64

linhas com IP sem ASN: 13

 as_handle          1
as_org             1
as_country_code    1
dtype: int64


In [6]:
# --- 6) Salva
full.to_parquet(DATA_FINAL / "full_dns_enriched.parquet", index=False)
full.to_csv(DATA_FINAL / "full_dns_enriched.csv", index=False)

In [7]:
from src.pipeline_audit import run_audit
cls = run_audit(out_path="../data/analysis/pipeline_limitations_report.txt")  # devolve um DataFrame com a etapa de falha por empresa

AUDITORIA DO PIPELINE - LIMITAÇÕES E EMPRESAS NÃO RESOLVIDAS
Gerado em 2026-11-02 20:15

1. FUNIL: TAMANHO DE CADA TABELA INTERMEDIÁRIA
------------------------------------------------------------------------------
  raw (top_companies_revenue_20261102.csv): 11396 linhas
  01_data_after_yfinance.csv           :   10667 linhas | 10667 empresas
  ns_resolver/companies.parquet        :   10664 linhas | 10664 empresas | 10321 com apex | 10261 apex únicos
  ns_resolver/dns_status.parquet       :      29 domínios com falha registrada
  ns_resolver/ns_hosts.parquet         :   32024 pares (domínio, NS) | 10233 domínios | 10789 NS únicos
  ns_resolver/ns_ips.parquet           :   26456 linhas | 10789 NS | 18889 IPs únicos
  asn_resolver/ip_asn.parquet          :   18889 IPs | 1430 ASNs
  asn_resolver/asn_org_info.parquet    :    1430 ASNs com org
  full_dns_enriched.parquet            :   83539 linhas | 10635 empresas

2. CONSISTÊNCIA ENTRE TABELAS (itens órfãos = informação perdida entre etap

In [8]:
# --- 7) Salva só completos

# mantém só as linhas completamente resolvidas: NS -> IP -> ASN -> as_org

resolved_cols = ["ns", "ip", "asn", "as_org"]
full_resolved = pd.DataFrame()
full_unresolved = pd.DataFrame()

full_resolved = full.dropna(subset=resolved_cols).copy()
full_unresolved = full.drop(full_resolved.index)  # o que foi removido, para auditoria

print(f"linhas: {len(full)} -> {len(full_resolved)} "
      f"(removidas {len(full_unresolved)}, {len(full_unresolved) / len(full):.1%})")
print(f"empresas: {full['rank'].nunique()} -> {full_resolved['rank'].nunique()}")


# salva a base limpa
full_resolved.to_parquet(DATA_FINAL / "full_clean.parquet", index=False)


full_resolved.to_csv(DATA_FINAL / "full_clean.csv", index=False)




linhas: 83787 -> 76636 (removidas 7151, 8.5%)
empresas: 10664 -> 10283


In [9]:
from src.depara_as_org import DEPARA_AS_ORG

# de-para -> DataFrame indexado por as_org
dp = pd.DataFrame.from_dict(DEPARA_AS_ORG, orient="index",
                            columns=["as_org_grupo", "as_country_code_grupo"])
display(dp.head(1))
remap = pd.DataFrame()
remap = full_resolved.copy()

remap["dp_as_org"] = remap["as_org"]
remap["dp_as_country_code"] = remap["as_country_code"]

remap = remap.join(dp, on="dp_as_org")

remap["dp_as_org"] = remap["as_org_grupo"].fillna(remap["dp_as_org"])
remap["dp_as_country_code"] = remap["as_country_code_grupo"].fillna(remap["dp_as_country_code"])
remap["as_org_remapped"] = remap["as_org_grupo"].notna()

remap = remap.drop(columns=["as_org_grupo", "as_country_code_grupo"])

print("linhas alteradas:", remap["as_org_remapped"].sum(), f"({remap['as_org_remapped'].mean():.1%})")
print("as_org únicas: antes", remap["as_org"].nunique(), "-> depois", remap["dp_as_org"].nunique())
print("empresas com org remapeada:", remap.loc[remap["as_org_remapped"], "rank"].nunique())

remap.to_parquet(DATA_FINAL / "final_data.parquet", index=False)
remap.to_csv(DATA_FINAL / "final_data.csv", index=False)


,as_org_grupo,as_country_code_grupo
"Vercara, LLC",DigiCert,US


linhas alteradas: 24999 (32.6%)
as_org únicas: antes 1292 -> depois 1119
empresas com org remapeada: 3681


In [10]:
mask = remap["company_country"].notna() & remap["country"].notna() & (remap["company_country"] != remap["country"])

pares = (
    remap.loc[mask]
      .drop_duplicates("rank")
      .groupby(["company_name", "company_country", "country"])
      .size()
      .sort_values(ascending=False)
      .rename("empresas")
      .reset_index()
)
pares

,company_name,company_country,country,empresas
0,AMTD Digital,Hong Kong,France,1
1,AMTD IDEA Group,Hong Kong,France,1
2,ASMPT\n (ASM Pacific Technology),Hong Kong,Singapore,1
3,"Agora, Inc.",China,United States,1
4,Akanda,United Kingdom,Canada,1
...,...,...,...,...
166,Zepp Health,China,Netherlands,1
167,Zhuzhou CRRC Times Electric,Hong Kong,China,1
168,Zymeworks,Canada,United States,1
169,atai Life Sciences,Germany,United States,1
